# 2.1 — Apply AI Functions in Snowflake

**Exam domain:** Gen AI Functions (Domain 2.0) · **Weight:** 38%

## The problem this solves

Your support team writes about 4,000 free-text tickets a month. Someone wants them grouped
by topic, ranked by how angry the customer sounds, stripped of personal data before anyone
outside support reads them, and translated when they arrive in German. Today that is a person
reading tickets, or a Python service that pulls the text out of the warehouse, calls a model
over the network, and writes the answers back — a second system to secure, monitor and pay for.

AI functions remove that round trip. The model runs where the data already lives, and you call
it the way you call `UPPER()` — in the middle of a `SELECT`.

## What you will be able to do

- Call the AISQL functions (`AI_COMPLETE`, `AI_CLASSIFY`, `AI_SENTIMENT`, `AI_FILTER`, `AI_TRANSLATE`,
  `AI_EXTRACT`, `AI_REDACT`, `AI_EMBED`, `AI_SIMILARITY`) and read the exact shape each one returns
- Force a model into a fixed JSON shape with `response_format`, so downstream SQL can rely on it
- Aggregate across rows with `AI_AGG` and `AI_SUMMARIZE_AGG` instead of one call per row
- Chunk and embed text for retrieval, and budget tokens before you spend them
- Tell the current `AI_*` functions apart from the legacy `SNOWFLAKE.CORTEX.*` family

## Before you start

- Run `setup/dataset.sql` once. It creates `GENAI_STUDY.PUBLIC.SUPPORT_TICKETS` and
  `GENAI_STUDY.PUBLIC.PRODUCTS`.
- Your role needs the `USE AI FUNCTIONS` account privilege (granted to `PUBLIC` by default) **and**
  one of the `SNOWFLAKE.CORTEX_USER` or `SNOWFLAKE.AI_FUNCTIONS_USER` database roles.
- A running warehouse. AI functions bill model tokens *and* the warehouse time of the query.

📖 **Snowflake documentation for this notebook**
- [Snowflake Cortex AI Functions (AISQL) overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql)
- [Privileges and model access for Cortex AI Functions](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)
- [AI_COMPLETE (single string)](https://docs.snowflake.com/en/sql-reference/functions/ai_complete-single-string)
- [AI_CLASSIFY](https://docs.snowflake.com/en/sql-reference/functions/ai_classify)
- [AI_SENTIMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_sentiment)
- [Vector embeddings in Snowflake Cortex](https://docs.snowflake.com/en/user-guide/snowflake-cortex/vector-embeddings)


---

## How an AI function call actually behaves

An AI function is an ordinary SQL function with an unusual cost profile. Three consequences follow,
and most exam traps live in one of them.

**It runs once per row.** `AI_CLASSIFY(ticket_text, [...])` written twice in the same `SELECT` is two
model calls per row, not one. Filter the rows down *before* the function, not after.

**It returns a fixed shape, and the shape is not always what you expect.** Some functions return a
scalar, some return an OBJECT you have to navigate with `:` path syntax. Reading the wrong path
returns `NULL` — no error, no warning, just an empty column and a bill for the calls.

**It fails per row.** One malformed input can null out a column. Almost every AI function takes a
trailing `return_error_details` BOOLEAN: leave it off and a failing row is `NULL`; set it to `TRUE`
and you get `{"value": ..., "error": ...}` back so a batch job can log the failures and carry on.

### Return shapes, at a glance

| Function | Category | Returns |
|---|---|---|
| `AI_COMPLETE` | General | VARCHAR — or an OBJECT with `show_details => TRUE` |
| `AI_CLASSIFY` | Analysis | OBJECT `{"labels": ["billing"]}` — an array, with no confidence score |
| `AI_SENTIMENT` | Analysis | OBJECT `{"categories":[{"name":"overall","sentiment":"negative"}, ...]}` |
| `AI_FILTER` | Analysis | BOOLEAN — usable in `WHERE` and in `JOIN ... ON` |
| `AI_AGG` | Aggregation | VARCHAR |
| `AI_SUMMARIZE_AGG` | Aggregation | VARCHAR |
| `SNOWFLAKE.CORTEX.SUMMARIZE` | Text | VARCHAR — the scalar summariser for English text |
| `AI_TRANSLATE` | Text | VARCHAR |
| `AI_EXTRACT` | Extraction | OBJECT `{"error": ..., "response": {...}}`, plus `"scoring"` when `scores => TRUE` |
| `AI_REDACT` | Privacy | VARCHAR in redact mode, OBJECT with `spans` in detect mode |
| `AI_EMBED` | Vector | `VECTOR(FLOAT, n)` |
| `AI_MULTI_EMBED` | Vector | OBJECT holding an array of per-segment embeddings |
| `AI_SIMILARITY` | Vector | FLOAT from −1 to 1 |
| `AI_TRANSCRIBE` | Audio/video | JSON with `audio_duration`, `text`, and `segments` |
| `AI_PARSE_DOCUMENT` | Document | OBJECT `{"content": ...}` or `{"pages": [...]}` |
| `AI_COUNT_TOKENS` | Helper | INTEGER |
| `SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER` | Helper | ARRAY of chunks |
| `TO_FILE` / `PROMPT` | Helper | FILE reference / prompt object |

→ [More on the AISQL function family](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql)

### Who is allowed to call them

```sql
GRANT USE AI FUNCTIONS ON ACCOUNT TO ROLE <your_role>;          -- account privilege
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE <your_role>;  -- or SNOWFLAKE.AI_FUNCTIONS_USER
```

Both halves are required. The account privilege is granted to `PUBLIC` by default, so the database
role is usually the piece that is missing when a call fails with a privilege error.

The two database roles are not interchangeable. `AI_FUNCTIONS_USER` covers the **scalar** AI functions
— every Cortex AI function *except* the aggregates `AI_AGG` and `AI_SUMMARIZE_AGG`. `CORTEX_USER` is
the broader role and covers those too. A narrower third role, `CORTEX_EMBED_USER`, covers only the
embedding functions (`AI_EMBED`, `EMBED_TEXT_768`, `EMBED_TEXT_1024`) — useful when you want a RAG
pipeline role that can vectorise text but cannot generate any.

→ [More on privileges and model access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)


---

## 1. AI_COMPLETE — the general-purpose call

Use `AI_COMPLETE` when no task-specific function fits: rewriting, drafting, reasoning over a row,
answering an open question. When a task-specific function *does* fit — classification, sentiment,
extraction — prefer it. Those functions return a typed shape, so you are not parsing prose.

```sql
AI_COMPLETE( <model>, <prompt>
             [ , <model_parameters>, <response_format>, <show_details> ] )
```

| Argument | What it does |
|---|---|
| `model` | The model name as a string. Which names are valid depends on your region and your grants — list them with `SHOW CORTEX BASE MODELS IN SCHEMA SNOWFLAKE.MODELS` rather than trusting a hard-coded list |
| `model_parameters` | OBJECT: `temperature` (0–1, default **0**), `top_p` (0–1, default **0**), `max_tokens` (default **4096**, counts *output* only), `guardrails` (default **FALSE**) |
| `response_format` | Its **own positional argument**, not a key inside `model_parameters`. Either a JSON schema object, or a SQL type literal starting with `TYPE` whose top level is an `OBJECT` |
| `show_details` | BOOLEAN. `TRUE` returns `{choices, created, model, usage}`; with a `response_format` set, `structured_output` replaces `choices` |

A *warehouse* here is just the compute cluster running your query — the AI call is billed in model
tokens on top of that warehouse time.

### Other input shapes

A **FILE** is a reference to a staged file, produced by `TO_FILE('@stage','name')`. `PROMPT()` builds
a template object whose `{0}`, `{1}` placeholders are filled by columns or FILEs, which is how you get
several inputs into one call:

```sql
AI_COMPLETE('claude-sonnet-5',
            PROMPT('Are {0} and {1} the same product?',
                   TO_FILE('@imgs','a.png'), TO_FILE('@imgs','b.png')));
```

There is also a three-argument single-file form, `AI_COMPLETE(<model>, <predicate>, <file>)`, for one
image and one question.

→ [More on PROMPT and multi-file prompts](https://docs.snowflake.com/en/sql-reference/functions/ai_complete-prompt-object)

### Structured output, and what it costs

`response_format` makes the model return JSON that matches a schema you declare, so the next SQL
statement can read `:urgency` without defensive parsing. The trade-off: the schema is sent with every
call and the constrained decoding produces more tokens, so a deep schema costs more than free text.
Declare the three fields you need, not the twenty you might.

### Multi-turn conversations

The `[{'role':'system','content':...}, {'role':'user','content':...}]` conversation array is documented
on the **legacy** `SNOWFLAKE.CORTEX.COMPLETE` page — roles `system`, `user`, `assistant`, and only one
system message, which must come first. The `AI_COMPLETE` reference documents a string prompt or a
`PROMPT()` object instead. Notebook 2.3 builds the chat pattern on both.

→ [More on AI_COMPLETE](https://docs.snowflake.com/en/sql-reference/functions/ai_complete-single-string)


In [ ]:
%%sql -r complete_examples_1
-- Example 1: Simple completion
SELECT
    ticket_id,
    AI_COMPLETE('llama3.1-8b',
        'Summarize this support ticket in one sentence: ' || ticket_text
    ) AS summary
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE ticket_id = 1002;


In [ ]:
%%sql -r complete_examples_2
-- Example 2: structured output.
-- response_format is its own positional argument and takes a JSON schema.
SELECT
    ticket_id,
    AI_COMPLETE(
        model  => 'llama3.1-8b',
        prompt => 'Extract urgency, issue_type and suggested_team from: ' || ticket_text,
        model_parameters => {'temperature': 0, 'max_tokens': 512},
        response_format  => {
            'type': 'json',
            'schema': {
                'type': 'object',
                'properties': {
                    'urgency':        {'type': 'string'},
                    'issue_type':     {'type': 'string'},
                    'suggested_team': {'type': 'string'}
                },
                'required': ['urgency', 'issue_type', 'suggested_team']
            }
        }
    ) AS structured_triage
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE language = 'en'
LIMIT 3;


In [ ]:
%%sql -r complete_examples_3
-- Equivalent using a SQL type literal:
--   response_format => TYPE OBJECT(urgency STRING, issue_type STRING, suggested_team STRING)

-- Example 3: pipeline-safe variant.
-- return_error_details => TRUE keeps one bad row from nulling the column silently.
SELECT
    ticket_id,
    AI_COMPLETE('llama3.1-8b', 'One-line summary: ' || ticket_text, NULL, NULL, FALSE, TRUE) AS safe_summary
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS;
-- Named-argument form is clearer:
-- AI_COMPLETE(model => 'llama3.1-8b', prompt => '...', return_error_details => TRUE)
--   -> {"value": "...", "error": null}

-- ============================================================
-- PYTHON EQUIVALENTS
--   Current:  from snowflake.snowpark.functions import ai_complete
--   Legacy (deprecated end 2026): from snowflake.cortex import complete
-- ============================================================


> ### ⚠️ Common misconceptions
>
> **"I will pass `response_format` inside `model_parameters`, like the other options."**
> `response_format` is a separate positional argument, fourth in the signature. Put it inside
> `model_parameters` and the model never sees a schema — you get ordinary prose back, and a
> downstream `:field` accessor on it returns `NULL`.
> → [AI_COMPLETE](https://docs.snowflake.com/en/sql-reference/functions/ai_complete-single-string)
>
> **"`TRY_COMPLETE` is the safe version of `AI_COMPLETE`."**
> `TRY_COMPLETE` exists, but only in the legacy namespace as `SNOWFLAKE.CORTEX.TRY_COMPLETE`, and it
> is marked for deprecation by the end of 2026. There is no `AI_TRY_COMPLETE`. The `AI_*` equivalent
> is `return_error_details => TRUE`, which returns `{"value": ..., "error": ...}` instead of hiding
> the failure behind a `NULL`.
> → [TRY_COMPLETE (legacy)](https://docs.snowflake.com/en/sql-reference/functions/try_complete-snowflake-cortex)
>
> **"`max_tokens` limits how much text I send."**
> It caps the *output*. Input length is bounded by the model's context window instead, and you
> estimate it with `AI_COUNT_TOKENS`. Setting `max_tokens` low to save money on a long document
> truncates the answer, not the prompt.
> → [AI_COUNT_TOKENS](https://docs.snowflake.com/en/sql-reference/functions/ai_count_tokens)


---

## 2. AI_CLASSIFY — put a row in a bucket

The problem: you have a category column that people filled in by hand, or not at all. You want every
row assigned to one of a fixed set of labels, consistently, without writing a hundred `LIKE` patterns.

```sql
AI_CLASSIFY( <input>, <list_of_categories>
             [, <config_object> ] [, <return_error_details> ] )
```

| Argument | What it does |
|---|---|
| `input` | Text, an image, a document, or a `PROMPT()` object |
| `list_of_categories` | ARRAY with **at least two unique values**. Either plain strings, or objects `{'label': 'billing', 'description': '...'}` where the description is 25 words or fewer. Labels are case-sensitive, and accuracy starts to drop past about twenty categories |
| `config_object` | `task_description` (50 words or fewer), `output_mode` — `'single'` (default) or `'multi'` — and `examples` for few-shot guidance |

It returns:

```json
{ "labels": ["billing"] }
```

Single-label mode always returns exactly one element; `output_mode => 'multi'` can return several.
Read the winner as `AI_CLASSIFY(...):labels[0]::VARCHAR`.

Reach for `task_description` when label names are ambiguous out of context ("Storage" — of files, or of
grain?), and for `examples` when the boundary between two labels is a judgement call you can only
demonstrate.

→ [More on AI_CLASSIFY](https://docs.snowflake.com/en/sql-reference/functions/ai_classify)


In [ ]:
%%sql -r classify_examples_1
-- Example 1: basic classification. The winning label is :labels[0], not :label.
SELECT
    ticket_id,
    AI_CLASSIFY(ticket_text, ['billing', 'technical', 'shipping'])                  AS clf,
    AI_CLASSIFY(ticket_text, ['billing', 'technical', 'shipping']):labels[0]::VARCHAR AS predicted,
    category                                                                        AS actual
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE language = 'en';


In [ ]:
%%sql -r classify_examples_2
-- Example 2: config object with task_description
SELECT
    ticket_id,
    AI_CLASSIFY(
        ticket_text,
        ['billing', 'technical', 'shipping', 'positive_feedback'],
        {'task_description': 'Classify customer support tickets for a SaaS cloud platform'}
    ):labels[0]::VARCHAR AS category
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS;


In [ ]:
%%sql -r classify_examples_3
-- Example 3: multi-label + labelled categories with descriptions
SELECT
    product_id,
    product_name,
    AI_CLASSIFY(
        description,
        [
          {'label': 'Security',        'description': 'access control, encryption, compliance'},
          {'label': 'Analytics',       'description': 'dashboards, reporting, BI'},
          {'label': 'Logistics',       'description': 'shipping, fulfilment, inventory'},
          {'label': 'Developer Tools', 'description': 'SDKs, CLIs, APIs'},
          {'label': 'Storage',         'description': 'object or block storage'}
        ],
        {'output_mode': 'multi'}
    ):labels AS ai_categories,
    category AS actual_category
FROM GENAI_STUDY.PUBLIC.PRODUCTS;


> ### ⚠️ Common misconceptions
>
> **"I will keep only the classifications the model was confident about, using the score it returns."**
> `AI_CLASSIFY` returns labels and nothing else — there is no `score` and no `confidence` key. A
> predicate like `AI_CLASSIFY(...):score > 0.8` compares `NULL` to a number, so it is never true and
> the query silently returns zero rows. If you genuinely need a number, get it from
> `AI_EXTRACT(..., scores => TRUE)` or from `AI_COMPLETE` with a `response_format` that declares one.
> → [AI_CLASSIFY](https://docs.snowflake.com/en/sql-reference/functions/ai_classify)
>
> **"`:label` is the obvious accessor for a classification result."**
> The key is `labels`, and it holds an array even in single-label mode. `:label` returns `NULL`;
> `:labels` returns `["billing"]`, which is not a VARCHAR. The accessor you want is
> `:labels[0]::VARCHAR`.
> → [AI_CLASSIFY](https://docs.snowflake.com/en/sql-reference/functions/ai_classify)
>
> **"One category is enough if I only care whether a ticket is about billing."**
> The array must contain at least two unique values. For a yes/no question, `AI_FILTER` is the right
> function — it returns a BOOLEAN and works directly in `WHERE`.
> → [AI_FILTER](https://docs.snowflake.com/en/sql-reference/functions/ai_filter)


---

## 3. AI_SENTIMENT and AI_FILTER

### AI_SENTIMENT

```sql
AI_SENTIMENT( <text> [, <categories> ] [, <return_error_details> ] )
```

`categories` is an optional ARRAY of up to **ten** aspect names, each at most **30 characters**. Pass
it when "is this ticket positive?" is too blunt a question — a customer can be happy with your support
and furious about delivery in the same paragraph.

It returns:

```json
{ "categories": [ { "name": "overall",  "sentiment": "negative" },
                  { "name": "delivery", "sentiment": "mixed"    } ] }
```

The `"overall"` record is always present. Each `sentiment` is one of `positive`, `negative`, `neutral`,
`mixed`, or `unknown` — the last meaning the aspect was not discussed.

Because the result is an array rather than a fixed object, pulling out the overall label means
flattening it and filtering on `name = 'overall'`. That is what the next cell does.

### AI_FILTER

```sql
AI_FILTER( <input> [, <return_error_details> ] )                 -- predicate embedded in the text
AI_FILTER( <predicate>, <file> [, <return_error_details> ] )     -- image input
AI_FILTER( PROMPT('<template>', <col1>, <col2>, ...) [, ... ] )  -- several columns
```

It returns a BOOLEAN, so it drops straight into a `WHERE` clause — and also into `JOIN ... ON`, which
lets you join two tables on a semantic condition rather than an equality. That is powerful and
expensive: a join predicate is evaluated across the cross product, so restrict both sides first.

→ [More on AI_FILTER](https://docs.snowflake.com/en/sql-reference/functions/ai_filter)


In [ ]:
%%sql -r sentiment_filter_1
-- Example 1: AI_SENTIMENT returns an array of category records.
SELECT
    ticket_id,
    category,
    AI_SENTIMENT(ticket_text) AS sentiment_obj,
    -- pull the always-present "overall" record out of the categories array
    (SELECT c.value:sentiment::VARCHAR
       FROM LATERAL FLATTEN(AI_SENTIMENT(ticket_text):categories) c
      WHERE c.value:name::VARCHAR = 'overall') AS overall_sentiment
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE language = 'en';


In [ ]:
%%sql -r sentiment_filter_2
-- Example 2: aspect-based sentiment (up to 10 categories, <=30 chars each)
SELECT
    ticket_id,
    AI_SENTIMENT(ticket_text, ['billing', 'delivery', 'support quality']) AS aspect_sentiment
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE language = 'en';


In [ ]:
%%sql -r sentiment_filter_3
-- Example 3: AI_FILTER — predicate is embedded in the text argument
SELECT ticket_id, ticket_text
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE AI_FILTER(
    'Does the following ticket describe a security breach, unauthorized account access, '
    || 'or compromised credentials? Ticket: ' || ticket_text
);


In [ ]:
%%sql -r sentiment_filter_4
-- Example 4: AI_FILTER with PROMPT() across two columns
SELECT product_name, category
FROM GENAI_STUDY.PUBLIC.PRODUCTS
WHERE AI_FILTER(PROMPT('Is the product {0} described by {1} relevant to regulatory compliance?',
                       product_name, description));


> ### ⚠️ Common misconceptions
>
> **"I can rank tickets by how negative they are with `AI_SENTIMENT(x):scores:negative > 0.7`."**
> This is the single most common error in Cortex study material, and it fails quietly.
> `AI_SENTIMENT` returns categorical labels — `{"categories":[{"name":"overall","sentiment":"negative"}]}`
> — with no numeric scores anywhere in the object. `:scores:negative` resolves to `NULL`, `NULL > 0.7`
> is never true, and your "most negative tickets" report comes back empty with no error to explain why.
> Filter on the label instead: `... WHERE sentiment = 'negative'`.
> → [AI_SENTIMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_sentiment)
>
> **"There must be a numeric sentiment score somewhere — I have definitely seen one."**
> You have. It belongs to the *legacy* function. `SNOWFLAKE.CORTEX.SENTIMENT(<text>)` returns a FLOAT
> from −1 to 1, banded as 0.5 to 1 positive, −0.5 to 0.5 neutral, and −0.5 to −1 negative. If an exam
> question hands you a float threshold on sentiment, the function in play is the legacy one.
> → [SENTIMENT (legacy)](https://docs.snowflake.com/en/sql-reference/functions/sentiment-snowflake-cortex)
>
> **"`AI_SENTIMENT(text, ['delivery'])` returns only the delivery aspect."**
> It returns delivery *and* `overall`, because the overall record is always included. Code that assumes
> `categories[0]` is your first requested aspect will read the wrong element.
> → [AI_SENTIMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_sentiment)


---

## 4. Summarising one row versus summarising a table

Two different questions, two different functions, and mixing them up is a classic mistake.

- **One summary per row** — `SNOWFLAKE.CORTEX.SUMMARIZE(<text>)` returns a VARCHAR. It is documented
  for English input. There is no `AI_SUMMARIZE` scalar function.
- **One summary per group** — `AI_SUMMARIZE_AGG(<expr>)` is an aggregate. Put it with a `GROUP BY` and
  you get one summary per category.
- **One answer per group, to your own question** — `AI_AGG(<expr>, <instruction>)` is the open-ended
  aggregate. The instruction works best written as a command ("Identify the top three pain points"),
  not as a question.

Both aggregates support datasets **larger than the model's context window**, which a per-row summariser
cannot do. That is the reason to reach for them rather than concatenating rows with `LISTAGG` and
hoping the result fits.

The cost trade-off: an aggregate is one expensive call per group; a scalar summariser is one cheap call
per row. For 50,000 rows in 5 categories, the aggregate is dramatically cheaper — and you lose the
per-row detail entirely.

→ [More on AI_AGG](https://docs.snowflake.com/en/sql-reference/functions/ai_agg) ·
[More on AI_SUMMARIZE_AGG](https://docs.snowflake.com/en/sql-reference/functions/ai_summarize_agg)

> **Privilege note.** `SNOWFLAKE.AI_FUNCTIONS_USER` covers scalar functions only. `AI_AGG` and
> `AI_SUMMARIZE_AGG` need `SNOWFLAKE.CORTEX_USER`.


In [ ]:
%%sql -r summarize_agg_1
-- ============================================================
-- SUMMARISATION
-- There is NO documented scalar AI_SUMMARIZE function.
--   * scalar   : SNOWFLAKE.CORTEX.SUMMARIZE(<text>) -> VARCHAR   (English text)
--   * aggregate: AI_SUMMARIZE_AGG(<expr>)           -> VARCHAR   (also works on one string)
-- AI_SUMMARIZE_AGG handles datasets LARGER than the model context window.
-- ============================================================

-- Example 1: per-row summary (scalar)
SELECT
    ticket_id,
    priority,
    SNOWFLAKE.CORTEX.SUMMARIZE(ticket_text) AS ticket_summary
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE status = 'open';


In [ ]:
%%sql -r summarize_agg_2
-- Example 2: one summary per category (aggregate + GROUP BY)
SELECT
    category,
    COUNT(*)                      AS ticket_count,
    AI_SUMMARIZE_AGG(ticket_text) AS category_summary
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
GROUP BY category;


In [ ]:
%%sql -r summarize_agg_3
-- AI_AGG — open-ended aggregation with your own instruction
-- Signature: AI_AGG( <expr>, <instruction> ) -> VARCHAR   [aggregate; exceeds context window safely]
SELECT
    AI_AGG(
        review_summary,
        'Identify the top 3 most common customer pain points across all product reviews. Be concise.'
    ) AS pain_points
FROM GENAI_STUDY.PUBLIC.PRODUCTS;

-- RBAC note: SNOWFLAKE.AI_FUNCTIONS_USER does NOT cover AI_AGG or AI_SUMMARIZE_AGG.
-- Those two require SNOWFLAKE.CORTEX_USER.


---

## 5. AI_TRANSLATE

```sql
AI_TRANSLATE( <text>, <source_language>, <target_language> [, <return_error_details> ] )
```

The argument that trips people up is the source language. To auto-detect it, pass an **empty string**
`''`. There is no `'auto'` keyword — pass one and you are asking to translate *from* a language code
that does not exist.

The normal reason to translate first is that most downstream AI functions are tuned for English. A
pipeline that normalises every row to English before classifying gets more consistent labels than one
that classifies in five languages — at the cost of an extra model call per non-English row, and of
whatever nuance the translation loses.

→ [More on AI_TRANSLATE](https://docs.snowflake.com/en/sql-reference/functions/ai_translate)


In [ ]:
%%sql -r translate_examples_1
-- AI_TRANSLATE
-- Signature: AI_TRANSLATE( <text>, <source_language>, <target_language> [, <return_error_details> ] )
-- Auto-detect the source language by passing an empty string ''.

-- Example 1: translate all non-English tickets to English
SELECT
    ticket_id,
    language,
    ticket_text                          AS original,
    AI_TRANSLATE(ticket_text, '', 'en')  AS english_translation   -- '' = auto-detect
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE language <> 'en';


In [ ]:
%%sql -r translate_examples_2
-- Example 2: normalise every ticket to English for downstream AI
SELECT
    ticket_id,
    CASE
        WHEN language = 'en' THEN ticket_text
        ELSE AI_TRANSLATE(ticket_text, '', 'en')
    END AS normalized_text
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS;


In [ ]:
%%sql -r translate_examples_3
-- Example 3: known source language -> French
SELECT
    ticket_id,
    AI_TRANSLATE(resolution_notes, 'en', 'fr') AS notes_french
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE resolution_notes IS NOT NULL;


> ### 🤔 Stop and think
>
> - Redacting before classification means the model never sees a customer name. It also means the model
>   never sees that two tickets are about the same person. Which of those matters more for your use case,
>   and who in your organisation gets to decide?
> - `AI_FILTER` in a `WHERE` clause reads beautifully and runs the model on every row that reaches it.
>   Where in a query would you put a cheap conventional predicate so that the expensive one sees fewer
>   rows — and what would you lose if that cheap predicate is slightly wrong?
> - A per-row `SNOWFLAKE.CORTEX.SUMMARIZE` costs roughly N calls; an `AI_AGG` over the same data costs
>   roughly one. If the per-row summaries are only ever read in aggregate, is the detail worth the bill —
>   and how would you find out without running both for a month?


---

## 6. AI_EXTRACT and AI_REDACT

### AI_EXTRACT — pull named fields out of text or a file

The problem: the invoice number is in the ticket body, in a different place every time. You want a
column, not a regex.

```sql
-- text input
AI_EXTRACT( <text>, <responseFormat> )
AI_EXTRACT( text => <text>, responseFormat => <fmt>, [ scores => TRUE|FALSE ] )

-- file input, straight from a stage; no separate parse step is required
AI_EXTRACT( file => TO_FILE('@stage','doc.pdf'), responseFormat => <fmt>,
            [ config => <config_object> ], [ scores => TRUE|FALSE ] )

-- a fine-tuned extraction model
AI_EXTRACT( model => 'db.schema.my_tuned_model', file => <file>, [ responseFormat => <fmt> ] )
```

A *stage* is Snowflake's named location for files — internal (Snowflake-managed) or external (your own
cloud bucket).

**`responseFormat` accepts four shapes:**

| Shape | Example |
|---|---|
| Object of questions | `{'name': 'What is the customer name?', 'total': 'What is the total due?'}` |
| Array of questions | `['What is the invoice number?', 'What is the due date?']` |
| Array of `[label, question]` pairs | `[['inv_no','What is the invoice number?'], ['due','When is it due?']]` |
| JSON schema | `{'schema': {'type':'object','properties': {...}}}` — required for **lists** and **tables**; table schemas add `column_ordering` |

Every response is wrapped:

```json
{ "error": null, "response": { "name": "Acme Corp", "total": "2499.00" } }
```

With `scores => TRUE` a `"scoring"` block is added alongside.

**Limits:** at most 100 entity questions or 10 table questions per call; 512 output tokens per entity
question and 4,096 for tables; documents no more than 125 pages and under 100 MB. File input accepts
PDF, PNG, PPTX, PPT, EML, DOC, DOCX, JPEG, JPG, HTM, HTML, TEXT, TXT, TIF, TIFF, BMP, GIF, WEBP and MD.
Role: `SNOWFLAKE.CORTEX_USER`.

→ [More on AI_EXTRACT](https://docs.snowflake.com/en/sql-reference/functions/ai_extract)

---

### AI_REDACT — strip PII before anything else touches the text

```sql
AI_REDACT( <input> [, <categories> ] [, <return_error_details> ] [, <mode> ] )
```

| Argument | What it does |
|---|---|
| `input` | A VARCHAR that may contain personal data |
| `categories` | An **ARRAY of strings** naming the PII types to act on. Omit it to cover every supported category |
| `mode` | `'redact'` (default) or `'detect'`; the value is case-insensitive |

- **`'redact'`** returns a VARCHAR with each detected item replaced by a category placeholder such as
  `[NAME]` — the placeholder tells you *what* was removed, which a blanket `[REDACTED]` would not.
- **`'detect'`** returns an OBJECT with a `spans` array of `{category, start, end, text}`. Use it to
  audit what is in a column, or to build your own masking that a downstream system can reverse.

Put `AI_REDACT` first in any chain, before the text reaches a second model or leaves the account.

→ [More on AI_REDACT](https://docs.snowflake.com/en/sql-reference/functions/ai_redact)


In [ ]:
%%sql -r extract_examples_1
-- Example 1: extract structured fields. Read results through the :response wrapper.
SELECT
    ticket_id,
    AI_EXTRACT(
        ticket_text,
        {
            'order_id':      'What order, invoice or case reference ID is mentioned?',
            'error_code':    'What error or incident code is mentioned (e.g. APP-XXX, INC-XXXX)?',
            'dollar_amount': 'What monetary amount is mentioned?'
        }
    ) AS extracted            -- {"error": null, "response": {...}}
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE language = 'en';


In [ ]:
%%sql -r extract_examples_2
-- Example 2: read one field out of the wrapper
WITH extracted AS (
    SELECT
        ticket_id,
        AI_EXTRACT(ticket_text, {'order_id': 'What is the order or invoice reference ID?'}) AS fields
    FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
    WHERE language = 'en'
)
SELECT
    ticket_id,
    fields:response:order_id::VARCHAR AS order_id      -- NOTE the :response level
FROM extracted
WHERE fields:response:order_id IS NOT NULL;


In [ ]:
%%sql -r extract_examples_3
-- Example 3: confidence scores + array-of-questions form
SELECT
    ticket_id,
    AI_EXTRACT(
        text           => ticket_text,
        responseFormat => [['version', 'What software version number is mentioned?'],
                           ['device',  'What hardware device is mentioned?']],
        scores         => TRUE
    ) AS tech_details   -- {"error":null, "response":{...}, "scoring":{"scores":{...}}}
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE category = 'technical';


In [ ]:
%%sql -r redact_examples_1
-- Example 1: Redact ALL supported PII categories (placeholders look like [NAME], [ADDRESS])
SELECT
    ticket_id,
    ticket_text            AS original,
    AI_REDACT(ticket_text) AS redacted
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE ticket_id = 1010;


In [ ]:
%%sql -r redact_examples_2
-- Example 2: redact specific categories. The categories argument is an ARRAY of strings.
SELECT
    ticket_id,
    AI_REDACT(ticket_text, ['EMAIL', 'PHONE_NUMBER']) AS contact_redacted
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
LIMIT 5;


In [ ]:
%%sql -r redact_examples_3
-- Example 3: detect mode — audit what PII exists without changing the text
SELECT
    ticket_id,
    AI_REDACT(ticket_text, NULL, FALSE, 'detect') AS pii_spans
    -- {"spans":[{"category":"EMAIL","start":42,"end":61,"text":"a@b.com"}, ...]}
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
LIMIT 5;


In [ ]:
%%sql
-- Example 4: compliance-safe copy of the ticket table
CREATE OR REPLACE TABLE GENAI_STUDY.PUBLIC.TICKETS_SAFE AS
SELECT
    ticket_id, created_at, category, status, priority, language, satisfaction_score,
    AI_REDACT(ticket_text) AS safe_ticket_text
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS;


In [ ]:
%%sql -r redact_examples_5
SELECT * FROM GENAI_STUDY.PUBLIC.TICKETS_SAFE LIMIT 5;


> ### ⚠️ Common misconceptions
>
> **"`AI_EXTRACT(text, {'total': '...'}):total` gives me the total."**
> Every result is wrapped in `{"error": ..., "response": {...}}`, so the field lives one level down:
> `:response:total::VARCHAR`. Skipping the `:response` level returns `NULL` for every row, which looks
> exactly like a model that failed to find anything.
> → [AI_EXTRACT](https://docs.snowflake.com/en/sql-reference/functions/ai_extract)
>
> **"I must run `AI_PARSE_DOCUMENT` before `AI_EXTRACT` on a PDF."**
> You do not. `AI_EXTRACT` takes a FILE directly. Parse first only when you also want the document's
> full text or markdown for another purpose — chunking it for retrieval, for instance. Parsing you do
> not need is a second billed pass over every page.
> → [AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document)
>
> **"`AI_REDACT(text, {'entities': ['EMAIL']})` limits it to e-mail addresses."**
> `categories` is a plain ARRAY — `AI_REDACT(text, ['EMAIL'])`. An object in that position is the wrong
> type for the argument, so the call errors rather than quietly redacting everything.
> → [AI_REDACT](https://docs.snowflake.com/en/sql-reference/functions/ai_redact)


---

## 7. AI_EMBED and the vector functions

An *embedding* is a fixed-length array of floats that positions a piece of text in a space where
"close together" means "about the same thing". That is what makes semantic search possible: you embed
the query, embed the documents, and compare numbers instead of matching words.

```sql
AI_EMBED( <model>, <input> )    -- input: VARCHAR, or a FILE for image models
```

Role: `SNOWFLAKE.CORTEX_USER` or the narrower `SNOWFLAKE.CORTEX_EMBED_USER`.

| Model | Dimensions | Context window | Languages |
|---|---|---|---|
| `snowflake-arctic-embed-m-v1.5` | 768 | 512 | English |
| `snowflake-arctic-embed-m` | 768 | 512 | English |
| `e5-base-v2` | 768 | 512 | English |
| `snowflake-arctic-embed-l-v2.0` | 1024 | 512 | Multilingual |
| `nv-embed-qa-4` | 1024 | 512 | English |
| `voyage-multilingual-2` | 1024 | 32,000 | Multilingual |

`voyage-multimodal-3` is the image model. Note the context windows: 512 tokens is the usual limit, so
anything longer must be chunked before it is embedded — see the helper functions below.

### The VECTOR data type

`VECTOR(INT | FLOAT, <dimension>)`, where dimension is a positive integer up to **4,096**. Comparing
two vectors with `<` or `>` is byte-wise and lexicographic: deterministic, and meaningless as a measure
of similarity. Always use a vector function.

| Function | Use case | Higher means more similar? |
|---|---|---|
| `VECTOR_COSINE_SIMILARITY(v1, v2)` | Semantic similarity — the retrieval default | Yes |
| `VECTOR_L2_DISTANCE(v1, v2)` | Euclidean distance | No — lower is closer |
| `VECTOR_L1_DISTANCE(v1, v2)` | Manhattan distance | No — lower is closer |
| `VECTOR_INNER_PRODUCT(v1, v2)` | Dot product; meaningful on normalised vectors | Yes |

→ [More on vector embeddings and distance functions](https://docs.snowflake.com/en/user-guide/snowflake-cortex/vector-embeddings)

### AI_SIMILARITY — comparison without managing vectors

```sql
AI_SIMILARITY( <input1>, <input2> )
AI_SIMILARITY( <input1>, <input2>, <config_object> )   -- {'model': '...'}
```

The model name goes **inside the config object**, never as a bare third string. Defaults are
`snowflake-arctic-embed-l-v2.0` for text and `voyage-multimodal-3` for images. It returns a float from
−1 to 1, and it cannot compare text against an image.

The trade-off against storing vectors yourself: `AI_SIMILARITY` embeds both inputs on every call. For a
handful of pairs that is the simplest thing that works. Across a table, or for repeated searches over
the same corpus, embed once into a `VECTOR` column and compare with `VECTOR_COSINE_SIMILARITY` instead.

→ [More on AI_SIMILARITY](https://docs.snowflake.com/en/sql-reference/functions/ai_similarity)


In [ ]:
%%sql
-- Example 1: Populate the embeddings column on PRODUCTS
UPDATE GENAI_STUDY.PUBLIC.PRODUCTS
SET embedding = AI_EMBED('snowflake-arctic-embed-m-v1.5', description);


In [ ]:
%%sql -r embed_vector_examples_2
-- Example 2: Semantic search over stored vectors
WITH query_vec AS (
    SELECT AI_EMBED('snowflake-arctic-embed-m-v1.5',
                    'real-time data streaming with ML model inference') AS q
)
SELECT
    p.product_id, p.product_name, p.category,
    VECTOR_COSINE_SIMILARITY(p.embedding, qv.q) AS similarity
FROM GENAI_STUDY.PUBLIC.PRODUCTS p
CROSS JOIN query_vec qv
ORDER BY similarity DESC
LIMIT 3;


In [ ]:
%%sql -r embed_vector_examples_3
-- Example 3: AI_SIMILARITY with an explicit model, supplied in the config object.
SELECT
    a.product_name AS product_a,
    b.product_name AS product_b,
    AI_SIMILARITY(a.description, b.description,
                  {'model': 'snowflake-arctic-embed-l-v2.0'}) AS similarity
FROM GENAI_STUDY.PUBLIC.PRODUCTS a
JOIN GENAI_STUDY.PUBLIC.PRODUCTS b ON a.product_id < b.product_id
ORDER BY similarity DESC
LIMIT 5;


In [ ]:
%%sql -r embed_vector_examples_4
-- Example 4: distance vs similarity on the same pair
WITH embedded AS (
    SELECT ticket_id, AI_EMBED('snowflake-arctic-embed-m-v1.5', ticket_text) AS emb
    FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
    WHERE ticket_id IN (1001, 1007)
)
SELECT
    a.ticket_id AS ticket_a,
    b.ticket_id AS ticket_b,
    VECTOR_L2_DISTANCE(a.emb, b.emb)       AS l2_dist,        -- lower = closer
    VECTOR_COSINE_SIMILARITY(a.emb, b.emb) AS cosine_sim      -- higher = closer
FROM embedded a, embedded b
WHERE a.ticket_id < b.ticket_id;


> ### ⚠️ Common misconceptions
>
> **"`AI_SIMILARITY(a, b, 'snowflake-arctic-embed-l-v2.0')` picks the model."**
> The third argument is a config OBJECT: `{'model': 'snowflake-arctic-embed-l-v2.0'}`. A bare string
> there is the wrong type and the call fails.
> → [AI_SIMILARITY](https://docs.snowflake.com/en/sql-reference/functions/ai_similarity)
>
> **"`ORDER BY embedding DESC` will sort rows by similarity."**
> Vector comparison is byte-wise lexicographic. The query runs, returns a stable order, and that order
> has nothing to do with meaning — a wrong answer with no error attached. Compute a similarity column
> with `VECTOR_COSINE_SIMILARITY` and order by that.
> → [VECTOR data type](https://docs.snowflake.com/en/sql-reference/data-types-vector)
>
> **"Bigger embedding models are always better, so use 1024 dimensions everywhere."**
> Dimensions cost storage on every row and time on every comparison, and the English-only 768-dimension
> Arctic models are the documented default for Cortex Search. The real reasons to move to
> `snowflake-arctic-embed-l-v2.0` are multilingual content or measured recall, not the bigger number.
> → [Vector embeddings](https://docs.snowflake.com/en/user-guide/snowflake-cortex/vector-embeddings)


---

## 8. Helper functions: counting and chunking

### AI_COUNT_TOKENS — budget before you spend

```sql
AI_COUNT_TOKENS( '<function_name>', <input_text> [, <return_error_details> ] )
AI_COUNT_TOKENS( '<function_name>', '<model_name>', <input_text> [, <return_error_details> ] )
AI_COUNT_TOKENS( '<function_name>', <input_text>, <options> [, <return_error_details> ] )
AI_COUNT_TOKENS( '<function_name>', '<model_name>', <input_text>, <options> [, <return_error_details> ] )
```

The **first** argument is the AI function you are budgeting for, not the model. The model name is a
separate argument, supplied only for functions where you choose one. It returns an INTEGER estimate of
**input** tokens — so it tells you whether a document will fit, not what the answer will cost.

→ [More on AI_COUNT_TOKENS](https://docs.snowflake.com/en/sql-reference/functions/ai_count_tokens)

### SPLIT_TEXT_RECURSIVE_CHARACTER — chunk before you embed

```sql
SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
    '<text_to_split>',
    '<format>',        -- required: 'none' or 'markdown'
    <chunk_size>,      -- required: maximum characters per chunk
    [ <overlap> ],     -- optional; must be smaller than chunk_size
    [ <separators> ]   -- optional ordered ARRAY of split boundaries
) -- returns ARRAY
```

Two details are easy to get wrong: it lives in the `SNOWFLAKE.CORTEX.` namespace, and `format` is a
**required second argument**. Pass `'markdown'` and it will prefer to split on headers, code blocks and
tables rather than mid-section.

Overlap is the trade-off dial. More overlap means a sentence that straddles a boundary appears in both
chunks, so retrieval is less likely to cut an idea in half — and you store and embed more text for the
same document.

### TO_FILE and PROMPT

```sql
TO_FILE('@stage', 'filename')                    -- a FILE reference
PROMPT('Compare {0} with {1}', col_a, col_b)     -- a prompt object; arguments may be strings or FILEs
```

`PROMPT` substitutes the first expression for `{0}`, the second for `{1}`, and so on, and returns an
object that `AI_COMPLETE`, `AI_FILTER` and `AI_CLASSIFY` know how to consume. Note the stage
restrictions that apply across the file-taking AI functions: files on user stages, table stages, or
client-side-encrypted stages are not readable.

→ [More on PROMPT](https://docs.snowflake.com/en/sql-reference/functions/prompt) ·
[More on chunking](https://docs.snowflake.com/en/sql-reference/functions/split_text_recursive_character-snowflake-cortex)


In [ ]:
%%sql -r helper_functions_1
-- Example 1: gate rows on token count before embedding them.
SELECT
    ticket_id,
    AI_COUNT_TOKENS('AI_EMBED', 'snowflake-arctic-embed-m-v1.5', ticket_text) AS token_count,
    IFF(AI_COUNT_TOKENS('AI_EMBED', 'snowflake-arctic-embed-m-v1.5', ticket_text) <= 512,
        'SAFE_TO_EMBED', 'NEEDS_CHUNKING') AS embed_readiness
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
ORDER BY token_count DESC;


In [ ]:
%%sql -r helper_functions_2
-- Example 2: chunking. Note the SNOWFLAKE.CORTEX namespace and the required format argument.
SELECT
    product_id,
    product_name,
    f.index          AS chunk_index,
    f.value::VARCHAR AS chunk_text,
    AI_COUNT_TOKENS('AI_EMBED', 'snowflake-arctic-embed-m-v1.5', f.value::VARCHAR) AS chunk_tokens
FROM GENAI_STUDY.PUBLIC.PRODUCTS,
     LATERAL FLATTEN(
         SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(description, 'none', 200, 20, ['\n\n', '. ', ' '])
     ) f
WHERE product_id = 207;


In [ ]:
%%sql -r helper_functions_3
-- Example 3: full chunk-and-embed RAG preparation
SELECT
    p.product_id,
    f.index          AS chunk_index,
    f.value::VARCHAR AS chunk_text,
    AI_EMBED('snowflake-arctic-embed-m-v1.5', f.value::VARCHAR) AS chunk_embedding
FROM GENAI_STUDY.PUBLIC.PRODUCTS p,
     LATERAL FLATTEN(
         SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(p.description, 'none', 300, 30, ['\n\n', '. ', ' '])
     ) f;


In [ ]:
%%sql -r helper_functions_4
-- Example 4: markdown-aware chunking, via format => 'markdown'.
SELECT SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
    '# Title\nIntro text\n## Section A\nBody A\n## Section B\nBody B',
    'markdown', 100, 10
) AS md_chunks;


---

## Putting it together

**Scenario.** A fintech company stores thousands of customer complaint e-mails in
`SUPPORT_TICKETS.ticket_text`. Before anyone analyses them they must:

1. Remove all personal data
2. Classify each e-mail as `fraud`, `account_access`, `billing` or `general`
3. Keep only the e-mails whose overall sentiment is negative
4. Store the result for the compliance team

**Write a CTE chain that does all four.**

### Worked solution

```sql
CREATE OR REPLACE TABLE GENAI_STUDY.PUBLIC.COMPLIANCE_TICKETS AS
WITH redacted AS (
    SELECT
        ticket_id,
        created_at,
        AI_REDACT(ticket_text) AS clean_text          -- placeholders such as [NAME], [EMAIL]
    FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
    WHERE language = 'en'
),
analyzed AS (
    SELECT
        ticket_id,
        created_at,
        clean_text,
        AI_CLASSIFY(clean_text,
            ['fraud','account_access','billing','general']):labels[0]::VARCHAR AS complaint_type,
        (SELECT c.value:sentiment::VARCHAR
           FROM LATERAL FLATTEN(AI_SENTIMENT(clean_text):categories) c
          WHERE c.value:name::VARCHAR = 'overall')                             AS sentiment
    FROM redacted
)
SELECT *
FROM analyzed
WHERE sentiment = 'negative';
```

**Why it is shaped this way.** Redaction comes first so no personal data reaches a second model. Each
CTE adds exactly one transformation, and each AI function is written once per row — repeat
`AI_CLASSIFY(...)` in both the `SELECT` list and the `WHERE` clause and you pay for it twice.

The filter is on the sentiment **label**. There is no numeric score to threshold; see the misconception
box in section 3 for why a float predicate here returns an empty table rather than an error.

---

## Quick self-check

1. What exactly does `AI_CLASSIFY` return, and how do you read the winning label?
2. How do you auto-detect the source language in `AI_TRANSLATE`?
3. Which two functions are excluded from `SNOWFLAKE.AI_FUNCTIONS_USER`?
4. Where does the model name go in `AI_SIMILARITY`?
5. What is the second argument of `SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER`?
6. What does `AI_REDACT` put in place of a detected name?
7. What is the first argument of `AI_COUNT_TOKENS`?

*Answers: 1) `{"labels":[...]}`, read as `:labels[0]::VARCHAR`. 2) an empty string `''`. 3) `AI_AGG` and
`AI_SUMMARIZE_AGG`. 4) inside the optional config object, `{'model': '...'}`. 5) `format` — `'none'` or
`'markdown'`. 6) a category placeholder such as `[NAME]`. 7) the AI function name, for example
`'AI_COMPLETE'`.*


---

## The legacy `SNOWFLAKE.CORTEX.*` family

The exam guide lists names from both generations, so you need to recognise the older one even though
you would not build on it. Several of these pages now carry an explicit notice: "This legacy function
will be deprecated by the end of 2026."

| Legacy | Current equivalent | The difference that matters |
|---|---|---|
| `SNOWFLAKE.CORTEX.COMPLETE(model, prompt \| messages, options)` | `AI_COMPLETE` | The legacy page is where the `messages` conversation array is documented — roles `system`, `user`, `assistant`, one system message, first in the array |
| `SNOWFLAKE.CORTEX.TRY_COMPLETE` | `AI_COMPLETE(..., return_error_details => TRUE)` | Legacy returns `NULL` on failure; the `AI_*` form returns the error alongside the value |
| `SNOWFLAKE.CORTEX.SENTIMENT(text)` | `AI_SENTIMENT` | Legacy returns a **FLOAT from −1 to 1**. The current function returns labels |
| `SNOWFLAKE.CORTEX.SUMMARIZE(text)` | — | Still the documented **scalar** summariser; there is no `AI_SUMMARIZE`. Use `AI_SUMMARIZE_AGG` when you want one summary across many rows |
| `SNOWFLAKE.CORTEX.EXTRACT_ANSWER(doc, question)` | `AI_EXTRACT` | Legacy answers one question and returns a plain string. `AI_EXTRACT` takes a whole `responseFormat` and returns `{error, response}` |
| `SNOWFLAKE.CORTEX.EMBED_TEXT_768` / `_1024` | `AI_EMBED` | Same models, one function, dimension inferred from the model |

### The two sentiment functions, side by side

```sql
SNOWFLAKE.CORTEX.SENTIMENT('The delivery was late.')   -- a FLOAT, e.g. -0.72
AI_SENTIMENT('The delivery was late.')                 -- {"categories":[{"name":"overall","sentiment":"negative"}]}
```

Legacy bands: **0.5 to 1** positive · **−0.5 to 0.5** neutral · **−0.5 to −1** negative. The score
expresses polarity, not intensity — a −0.9 is not "twice as angry" as a −0.45.

**The rule for the exam:** asked what to use *for new work*, answer with the `AI_*` function. Shown
legacy syntax and asked what it does, answer with the legacy semantics — especially the float.

→ [More on the legacy SENTIMENT function](https://docs.snowflake.com/en/sql-reference/functions/sentiment-snowflake-cortex) ·
[More on the legacy COMPLETE function](https://docs.snowflake.com/en/sql-reference/functions/complete-snowflake-cortex)


In [ ]:
%%sql -r legacy_sentiment_float
-- Legacy SENTIMENT returns a FLOAT from -1 to 1. This is the only sentiment function with a number.
SELECT
    ticket_id,
    SNOWFLAKE.CORTEX.SENTIMENT(ticket_text) AS legacy_score,
    CASE
        WHEN SNOWFLAKE.CORTEX.SENTIMENT(ticket_text) >=  0.5 THEN 'positive'
        WHEN SNOWFLAKE.CORTEX.SENTIMENT(ticket_text) <= -0.5 THEN 'negative'
        ELSE 'neutral'
    END AS banded
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE language = 'en'
ORDER BY legacy_score;

In [ ]:
%%sql -r sentiment_legacy_vs_current
-- Side by side with the current function, which returns LABELS and no numbers
SELECT
    ticket_id,
    SNOWFLAKE.CORTEX.SENTIMENT(ticket_text) AS legacy_float,
    AI_SENTIMENT(ticket_text)               AS current_labels
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE language = 'en'
LIMIT 5;

In [ ]:
%%sql -r extract_answer_vs_ai_extract
-- Legacy EXTRACT_ANSWER: one question in, one string out
SELECT
    ticket_id,
    SNOWFLAKE.CORTEX.EXTRACT_ANSWER(ticket_text, 'What is the order or invoice number?') AS legacy_answer,
    AI_EXTRACT(ticket_text,
        {'order_no': 'What is the order or invoice number?'}):response:order_no::VARCHAR AS current_answer
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE language = 'en'
LIMIT 5;

In [ ]:
%%sql -r legacy_summarize
-- Legacy scalar summariser — still the documented one; there is no AI_SUMMARIZE
SELECT
    ticket_id,
    SNOWFLAKE.CORTEX.SUMMARIZE(ticket_text) AS summary
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE status = 'open';

---

## Check your understanding

Twelve questions on this notebook. Answer before expanding.

**1.** What is the default value of `temperature` in `AI_COMPLETE`'s `model_parameters`, and what does
`max_tokens` limit?

<details><summary>Show answer</summary>

`temperature` defaults to **0**, and so does `top_p`. `max_tokens` defaults to 4096 and caps the number
of **output** tokens only. A common assumption is that `temperature` defaults to something creative
like 0.7 — it does not, which is why repeated `AI_COMPLETE` calls on the same row tend to agree. If you
want variety you have to ask for it.

→ [AI_COMPLETE](https://docs.snowflake.com/en/sql-reference/functions/ai_complete-single-string)

</details>

**2.** Which two AI functions does the `SNOWFLAKE.AI_FUNCTIONS_USER` database role *not* cover?

<details><summary>Show answer</summary>

`AI_AGG` and `AI_SUMMARIZE_AGG` — the two aggregate functions. `AI_FUNCTIONS_USER` grants the scalar
functions only, so a role with just that database role gets a privilege error on an aggregate call even
though every other AI function works. `SNOWFLAKE.CORTEX_USER` covers both.

→ [Privileges and model access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)

</details>

**3.** What are the two required arguments of `SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER` after
the text itself, and in what order?

<details><summary>Show answer</summary>

`format` then `chunk_size` — `('<text>', '<format>', <chunk_size> [, <overlap> [, <separators> ]])`.
`format` is `'none'` or `'markdown'`. Calls written as `(text, 500, 50)` fail because 500 is being
passed where a format string is expected. The namespace matters too: the function is not available
unqualified.

→ [SPLIT_TEXT_RECURSIVE_CHARACTER](https://docs.snowflake.com/en/sql-reference/functions/split_text_recursive_character-snowflake-cortex)

</details>

**4.** A colleague runs this and gets zero rows, with no error. What is wrong?

```sql
SELECT ticket_id
FROM SUPPORT_TICKETS
WHERE AI_SENTIMENT(ticket_text):scores:negative > 0.7;
```

<details><summary>Show answer</summary>

`AI_SENTIMENT` returns `{"categories":[{"name":"overall","sentiment":"negative"}, ...]}` — labels, with
no `scores` object. The path resolves to `NULL`, and `NULL > 0.7` is never true, so every row is
filtered out silently. The fix is to flatten `:categories`, take the record whose `name` is `'overall'`,
and compare its `sentiment` to `'negative'`. If a numeric score is genuinely required, the legacy
`SNOWFLAKE.CORTEX.SENTIMENT` returns a FLOAT from −1 to 1.

→ [AI_SENTIMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_sentiment)

</details>

**5.** This query returns `NULL` in every row of `order_id`. Why?

```sql
SELECT AI_EXTRACT(ticket_text, {'order_id': 'What is the order ID?'}):order_id::VARCHAR AS order_id
FROM SUPPORT_TICKETS;
```

<details><summary>Show answer</summary>

The result is wrapped: `{"error": ..., "response": {"order_id": "..."}}`. The accessor has to go through
the wrapper — `:response:order_id::VARCHAR`. The tempting reading is that the model simply found nothing,
which sends people off to rewrite the prompt when the extraction was working all along.

→ [AI_EXTRACT](https://docs.snowflake.com/en/sql-reference/functions/ai_extract)

</details>

**6.** `AI_TRANSLATE(ticket_text, 'auto', 'en')` does not auto-detect the language. What does the
function actually expect?

<details><summary>Show answer</summary>

An empty string, `''`, in the `source_language` position. `'auto'` is not a language code and not a
documented keyword. Passing a known code such as `'de'` is the other valid option and is slightly more
reliable when you already store the language, as the `SUPPORT_TICKETS` table does.

→ [AI_TRANSLATE](https://docs.snowflake.com/en/sql-reference/functions/ai_translate)

</details>

**7.** You want one label per product, chosen from five categories, plus a confidence number so a human
can review the uncertain ones. Can `AI_CLASSIFY` give you both?

<details><summary>Show answer</summary>

No. It returns `{"labels": [...]}` and nothing else — no score field exists. To get a number you either
use `AI_EXTRACT` with `scores => TRUE`, or `AI_COMPLETE` with a `response_format` schema that declares
a numeric confidence field and accept that the model is self-reporting. A cheaper alternative is often
to skip the score and route ambiguous cases by rule: if `output_mode => 'multi'` returns more than one
label, send it to a human.

→ [AI_CLASSIFY](https://docs.snowflake.com/en/sql-reference/functions/ai_classify)

</details>

**8.** A pipeline calls `AI_COMPLETE` over 200,000 rows and one malformed row nulls out a column with no
message. What would you change?

<details><summary>Show answer</summary>

Add `return_error_details => TRUE`. The function then returns `{"value": ..., "error": ...}` per row, so
the batch keeps running and you can select the rows where `:error IS NOT NULL` to see what happened.
Without it, a failure and a legitimately empty answer look identical. Note this is the `AI_*` mechanism —
`SNOWFLAKE.CORTEX.TRY_COMPLETE` is the legacy equivalent and simply returns `NULL`.

→ [AISQL overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql)

</details>

**9.** You need to compare each of 50,000 documents against 200 reference documents. Using
`AI_SIMILARITY` for every pair, what have you signed up for?

<details><summary>Show answer</summary>

Ten million calls, each of which re-embeds **both** sides — so roughly 20 million embeddings for 50,200
distinct documents. Embedding each document once into a `VECTOR` column with `AI_EMBED` and comparing
with `VECTOR_COSINE_SIMILARITY` does the same work with 50,200 model calls and pure arithmetic
afterwards. `AI_SIMILARITY` earns its place when the comparison is one-off and you have nowhere to put
the vectors.

→ [AI_SIMILARITY](https://docs.snowflake.com/en/sql-reference/functions/ai_similarity)

</details>

**10.** Your tickets average 900 tokens and you plan to embed them with
`snowflake-arctic-embed-m-v1.5`. What do you check first, and what are the options if it fails?

<details><summary>Show answer</summary>

That model's context window is 512 tokens, so check with
`AI_COUNT_TOKENS('AI_EMBED', 'snowflake-arctic-embed-m-v1.5', ticket_text)` before you spend anything.
For rows over the limit the choices are: chunk with `SPLIT_TEXT_RECURSIVE_CHARACTER` and store one
vector per chunk (better retrieval, more rows, more storage), or move to a model with a longer window
such as `voyage-multilingual-2` at 32,000 tokens (fewer rows, a bigger vector per row, and a different
model to evaluate).

→ [Vector embeddings](https://docs.snowflake.com/en/user-guide/snowflake-cortex/vector-embeddings)

</details>

**11.** For a monthly report of the top complaints per product category over 50,000 reviews, would you
use `SNOWFLAKE.CORTEX.SUMMARIZE` per row or `AI_AGG` with a `GROUP BY`? What does each cost you?

<details><summary>Show answer</summary>

`AI_AGG` with `GROUP BY category` — one call per category instead of 50,000, and it handles input larger
than the model's context window, which a per-row summariser cannot. What you give up is the per-row
summary: if anyone ever needs to read one review's gist on its own, that data no longer exists and
regenerating it means paying the 50,000 calls after all. The usual compromise is to aggregate for the
report and summarise per row only for the small set a human actually opens.

→ [AI_AGG](https://docs.snowflake.com/en/sql-reference/functions/ai_agg)

</details>

**12.** *Connecting to another domain.* You have redacted ticket text and now want business users to
search it in natural language. Which Snowflake feature takes over from the AI functions here, and what
does it need from the table you built?

<details><summary>Show answer</summary>

A **Cortex Search service** — covered in 2.2. It does the embedding, indexing, hybrid retrieval and
reranking that you would otherwise hand-build from `AI_EMBED` plus `VECTOR_COSINE_SIMILARITY`. It needs
change tracking enabled on the underlying objects so it can refresh incrementally, a warehouse and a
`TARGET_LAG` for that refresh, and any columns you want to filter on declared as `ATTRIBUTES`. One
governance consequence to carry over: a search service runs with **owner's rights**, so it can surface
text the querying role could not read directly — which is exactly why you redacted first.

→ [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)

</details>
